# Lab 9: Augmentation and Keras ImageDataGenerator


**University of Engineering and Technology Peshawar, Nowshera Campus**

**Course:** Machine Learning Lab

**Student Name:** Muhammad Ayub  
**Registration Number:** 22jzele0470

**Date:** May 14, 2026

---

## Table of Contents

1. [Import Libraries](#1-import-libraries)  
2. [Set Paths and Checkpoint](#2-set-paths-and-checkpoint)  
3. [Build the CNN Model](#3-build-the-cnn-model)  
4. [Compile the Model](#4-compile-the-model)  
5. [Data Generators with Augmentation](#5-data-generators-with-augmentation)  
6. [Model Training](#6-model-training)  
7. [Plot Training History](#7-plot-training-history)  
8. [Model Evaluation](#8-model-evaluation)  

---


## 1. Import Libraries

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import seaborn as sns

## 2. Set Paths and Checkpoint

In [ ]:
checkpoints = r'C:\Users\M Ayub\Downloads\ML_LAB\Computer vision\checkpoints_lab_9\ML Lab\lab13\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'

In [ ]:
train_dir = r'C:\Users\M Ayub\Downloads\ML_LAB\Computer vision\train'
validation_dir = r'C:\Users\M Ayub\Downloads\ML_LAB\Computer vision\validation'
test_dir = r'C:\Users\M Ayub\Downloads\ML_LAB\Computer vision\test'

## 3. Build the CNN Model

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu',
input_shape=(256, 256, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dropout(0.5))
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(4, activation='sigmoid'))

## 4. Compile the Model

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer=optimizers.RMSprop(lr=1e-4), metrics=['acc'])

## 5. Data Generators with Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
                                    rescale=1./255,
                                    rotation_range=40,
                                    width_shift_range=0.2,
                                    height_shift_range=0.2,
                                    shear_range=0.2,
                                    zoom_range=0.2,
                                    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
                                                    train_dir,
                                                    target_size=(256, 256),
                                                    batch_size=32,
                                                    class_mode='categorical')

validation_generator = test_datagen.flow_from_directory(
                                                        validation_dir,
                                                        target_size=(256, 256),
                                                        batch_size=32,
                                                        class_mode='categorical')

## 6. Model Training

In [ ]:
EpochCheckpoint = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
callbacks = [EpochCheckpoint]

In [ ]:
model_history = model.fit(train_generator,
                         validation_data =validation_generator,
                         steps_per_epoch=train_generator.n//train_generator.batch_size,
                         validation_steps = validation_generator.n//validation_generator.batch_size,
                         epochs = 10,
                         callbacks = callbacks)

## 7. Plot Training History

In [ ]:
acc = model_history.history['acc']
val_acc = model_history.history['val_acc']
loss = model_history.history['loss']
val_loss = model_history.history['val_loss']
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

## 8. Model Evaluation

In [ ]:
model = load_model(r'C:\Users\M Ayub\Downloads\ML_LAB\Computer vision\checkpoints_lab_9\ML Lab\lab13\E1-cp-0005-loss0.31.h5')
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(test_dir, target_size=(256, 256), batch_size=32, shuffle=False, class_mode='categorical')
label=test_generator.labels
preds=model.predict(test_generator)
pred = np.argmax(preds, axis = 1)
cm = confusion_matrix(label, pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,  display_labels=['Cercospora', 'common_rust','healthy', 'leaf_blight'])
disp.plot()
plt.show()

In [ ]:
print(classification_report(label, pred, target_names=['Cercospora', 'common_rust','healthy', 'leaf_blight']))

---

**GitHub Repository:**  
https://github.com/prince4775/8th-Semester-ML-and-DL-Lab